In [1]:
# Célula 1 — Imports
import geopandas as gpd
import pandas as pd
from pathlib import Path

In [2]:
# Célula 2 — Definir pasta base
pasta = Path(r"C:\Users\franc\OneDrive\Francisco\Profissional\MBA_Data_Science_ e_Analytics\00_TCC\06_Dados_base\GEO\2024_02_basegeo")

# Listar todos os shapefiles disponíveis
shapefiles = list(pasta.rglob("*.shp"))
print(f"Shapefiles encontrados: {len(shapefiles)}")
for shp in shapefiles:
    print(f"  {shp.name}")

Shapefiles encontrados: 3
  bf.shp
  gl.shp
  lf.shp


In [3]:
# Célula 3 — Carregar e inspecionar cada grupo
grupos = ["lf", "bf", "gl"]

for grupo in grupos:
    arquivos = [s for s in shapefiles if grupo in s.stem.lower()]
    if not arquivos:
        print(f"\n[{grupo.upper()}] Nenhum arquivo encontrado.")
        continue
    for arq in arquivos:
        print(f"\n{'='*60}")
        print(f"[{grupo.upper()}] {arq.name}")
        print('='*60)
        gdf = gpd.read_file(arq)
        print(f"Registros:  {len(gdf):,}")
        print(f"CRS:        {gdf.crs}")
        print(f"Geometria:  {gdf.geom_type.unique()}")
        print(f"\nColunas ({len(gdf.columns)}):")
        for col in gdf.columns:
            nulos = gdf[col].isna().sum()
            pct   = nulos / len(gdf) * 100
            print(f"  {col:<30} {str(gdf[col].dtype):<15} nulos: {nulos:,} ({pct:.1f}%)")
        print(f"\nAmostra (2 registros):")
        print(gdf.drop(columns='geometry').head(2).to_string())


[LF] lf.shp


Registros:  42,016
CRS:        PROJCS["TM-POA",GEOGCS["SIRGAS 2000",DATUM["Sistema_de_Referencia_Geocentrico_para_las_AmericaS_2000",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6674"]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",-51],PARAMETER["scale_factor",0.999995],PARAMETER["false_easting",300000],PARAMETER["false_northing",5000000],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
Geometria:  ['Polygon' 'MultiPolygon' None]

Colunas (6):
  SETOR                          int64           nulos: 0 (0.0%)
  QUARTEIRAO                     int64           nulos: 0 (0.0%)
  AREA                           float64         nulos: 0 (0.0%)
  NUMBLOCO                       object          nulos: 0 (0.0%)
  IDSMFLF                        int64           nulos: 0 (0.0%)
  geometry                       geometr

Registros:  155,705
CRS:        PROJCS["TM-POA",GEOGCS["SIRGAS 2000",DATUM["Sistema_de_Referencia_Geocentrico_para_las_AmericaS_2000",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6674"]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",-51],PARAMETER["scale_factor",0.999995],PARAMETER["false_easting",300000],PARAMETER["false_northing",5000000],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
Geometria:  ['Polygon' 'MultiPolygon' None]

Colunas (6):
  NUMBLOCO                       object          nulos: 1 (0.0%)
  SETOR                          int64           nulos: 0 (0.0%)
  QUARTEIRAO                     int64           nulos: 0 (0.0%)
  AREA                           float64         nulos: 0 (0.0%)
  IDBFLF_                        int64           nulos: 0 (0.0%)
  geometry                       geomet

Registros:  468
CRS:        PROJCS["TM-POA",GEOGCS["SIRGAS 2000",DATUM["Sistema_de_Referencia_Geocentrico_para_las_AmericaS_2000",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6674"]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",-51],PARAMETER["scale_factor",0.999995],PARAMETER["false_easting",300000],PARAMETER["false_northing",5000000],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
Geometria:  ['Polygon' 'MultiPolygon' None]

Colunas (5):
  SETOR                          int64           nulos: 0 (0.0%)
  QUARTEIRAO                     int64           nulos: 0 (0.0%)
  AREA                           float64         nulos: 0 (0.0%)
  NUMBLOCO                       object          nulos: 2 (0.4%)
  geometry                       geometry        nulos: 1 (0.2%)

Amostra (2 registros):
   SETOR  QUARTEIRA

In [4]:
# Célula 4 — Análise de sobreposições e geometrias nulas
import geopandas as gpd
import pandas as pd
from pathlib import Path

pasta = Path(r"C:\Users\franc\OneDrive\Francisco\Profissional\MBA_Data_Science_ e_Analytics\00_TCC\06_Dados_base\GEO\2024_02_basegeo")

lf = gpd.read_file(pasta / "lf.shp")
bf = gpd.read_file(pasta / "bf.shp")
gl = gpd.read_file(pasta / "gl.shp")

lf["CAMADA"] = "LF"
bf["CAMADA"] = "BF"
gl["CAMADA"] = "GL"

bf = bf.rename(columns={"IDBFLF_": "ID"})
lf = lf.rename(columns={"IDSMFLF": "ID"})
gl["ID"] = range(len(gl))

print("=" * 60)
print("1. GEOMETRIAS NULAS")
print("=" * 60)
for nome, gdf in [("LF", lf), ("BF", bf), ("GL", gl)]:
    nulos = gdf[gdf.geometry.isna()]
    print(f"\n[{nome}] {len(nulos)} geometrias nulas:")
    if len(nulos) > 0:
        print(nulos[["NUMBLOCO", "AREA", "SETOR", "QUARTEIRAO", "CAMADA"]].to_string())

print("\n" + "=" * 60)
print("2. SOBREPOSIÇÕES DENTRO DA MESMA CAMADA")
print("=" * 60)

def checar_sobreposicao_interna(gdf, nome, amostra=5):
    gdf_valido = gdf[gdf.geometry.notna() & gdf.geometry.is_valid].copy()
    joined = gpd.sjoin(gdf_valido, gdf_valido, how="inner", predicate="overlaps")
    joined = joined[joined.index != joined["index_right"]]
    joined["par"] = joined.apply(
        lambda r: tuple(sorted([r.name, r["index_right"]])), axis=1
    )
    joined = joined.drop_duplicates("par")
    print(f"\n[{nome}] Sobreposições internas: {len(joined)}")
    if len(joined) > 0:
        cols = ["NUMBLOCO_left", "AREA_left", "NUMBLOCO_right", "AREA_right"]
        disponiveis = [c for c in cols if c in joined.columns]
        print(joined[disponiveis].head(amostra).to_string())

checar_sobreposicao_interna(lf, "LF")
checar_sobreposicao_interna(bf, "BF")
checar_sobreposicao_interna(gl, "GL")

print("\n" + "=" * 60)
print("3. SOBREPOSIÇÕES ENTRE CAMADAS DISTINTAS")
print("=" * 60)

def checar_sobreposicao_entre(gdf_a, nome_a, gdf_b, nome_b, amostra=10):
    a = gdf_a[gdf_a.geometry.notna() & gdf_a.geometry.is_valid].copy()
    b = gdf_b[gdf_b.geometry.notna() & gdf_b.geometry.is_valid].copy()
    joined = gpd.sjoin(a, b, how="inner", predicate="overlaps")
    print(f"\n[{nome_a} × {nome_b}] Sobreposições: {len(joined)}")
    if len(joined) > 0:
        print(f"  Amostra ({min(amostra, len(joined))} registros):")
        cols = ["NUMBLOCO_left", "AREA_left", "NUMBLOCO_right", "AREA_right"]
        disponiveis = [c for c in cols if c in joined.columns]
        print(joined[disponiveis].head(amostra).to_string())

checar_sobreposicao_entre(lf, "LF", bf, "BF")
checar_sobreposicao_entre(lf, "LF", gl, "GL")
checar_sobreposicao_entre(bf, "BF", gl, "GL")

1. GEOMETRIAS NULAS

[LF] 2 geometrias nulas:
          NUMBLOCO         AREA  SETOR  QUARTEIRAO CAMADA
3337  007000200001  86395.64443     39           4     LF
3637  007000200001  86395.64443     39           4     LF

[BF] 1 geometrias nulas:
            NUMBLOCO  AREA  SETOR  QUARTEIRAO CAMADA
155688  007189300000   0.0      0           0     BF

[GL] 1 geometrias nulas:
    NUMBLOCO  AREA  SETOR  QUARTEIRAO CAMADA
383     None   0.0      0           0     GL

2. SOBREPOSIÇÕES DENTRO DA MESMA CAMADA



[LF] Sobreposições internas: 4516
   NUMBLOCO_left   AREA_left NUMBLOCO_right    AREA_right
10  007143450000  832.471390   002557680000   1495.507510
11  007150580000  747.112983   001922560000   2907.025494
12  007005090000  299.999809   002498780000  58819.340058
26  007027930000  126.457832   007027940000    128.468339
26  007027930000  126.457832   007027920000    126.384244



[BF] Sobreposições internas: 10364
   NUMBLOCO_left    AREA_left NUMBLOCO_right   AREA_right
38  001019320000  1534.659066   007185800000  7391.171599
55  001247570000   430.139371   000547910000   815.272730
71  001420110000   202.849527   001781270000   398.825029
83  001736570000   772.614572   007031170000   566.041710
98  001524530000   148.143945   001545000000   172.254079

[GL] Sobreposições internas: 78
   NUMBLOCO_left      AREA_left NUMBLOCO_right     AREA_right
1   007041110012   15863.674205   007041110008   21533.414663
6   001264370000    5845.072969   000000000000  259734.444241
10  001348790000  536262.867700   001224730001  349695.373408
12  000000000000   10715.234650   000000000000     184.445550
13  002068030000   35624.781271   007097530000       0.000000

3. SOBREPOSIÇÕES ENTRE CAMADAS DISTINTAS



[LF × BF] Sobreposições: 23798
  Amostra (10 registros):
   NUMBLOCO_left     AREA_left NUMBLOCO_right   AREA_right
0   001636850000  38232.252691   007071270000  1298.538872
0   001636850000  38232.252691   007071260000  1310.556043
1   007025580000   8035.048352   001811430000   612.074176
1   007025580000   8035.048352   001811450000   788.450701
1   007025580000   8035.048352   001208870000   599.853973
14  002622460001     91.619406   007022330003   158.491576
15  007149950000  43929.125111   007015620000   134.520748
15  007149950000  43929.125111   007015630000    80.731634
15  007149950000  43929.125111   007015640000   121.867038
20  007027370000    896.149344   007051750000  1916.873262



[LF × GL] Sobreposições: 3634
  Amostra (10 registros):
    NUMBLOCO_left    AREA_left NUMBLOCO_right     AREA_right
12   007005090000   299.999809   002557780000  228120.970858
38   007018250000   294.618728   001968590000   14180.742019
49   007116210000  5155.061570   007039420000    6298.662956
50   007019570002  2254.545257   000000000000   10715.234650
54   007019570003  1067.338892   000000000000   10715.234650
56   007019570001  8831.495605   000000000000   10715.234650
57   007019570004  3939.837734   000000000000   10715.234650
158  000000000000   197.226648   000000000000   27370.440952
158  000000000000   197.226648   000000000000   34831.335971
163  002107110002    70.001245   002107110001     502.568763



[BF × GL] Sobreposições: 3709
  Amostra (10 registros):
      NUMBLOCO_left   AREA_left NUMBLOCO_right    AREA_right
14102  002449410000  146.387174   001102390000  78963.457860
16848  001656650000  152.183452   001102390000  78963.457860
17027  001326350000  618.988455   001321090000    199.784311
20425  001368940000  279.101389   001102390000  78963.457860
20444  007003720000  170.252157   001102390000  78963.457860
20549  001656890000  178.152563   001102390000  78963.457860
20550  001656910000  207.567367   001102390000  78963.457860
20561  001656720000  692.978937   001102390000  78963.457860
21000  002556270000  282.735861   001368740000   1666.073237
21004  002653460000  166.190555   001968590000  14180.742019


In [5]:
import geopandas as gpd
from pathlib import Path

pasta = Path(r"C:\Users\franc\OneDrive\Francisco\Profissional\MBA_Data_Science_ e_Analytics\00_TCC\06_Dados_base\GEO\2024_02_basegeo")

lf = gpd.read_file(pasta / "lf.shp")
bf = gpd.read_file(pasta / "bf.shp")

lf_val = lf[lf.geometry.notna() & lf.geometry.is_valid].copy()
bf_val = bf[bf.geometry.notna() & bf.geometry.is_valid].copy()

joined = gpd.sjoin(lf_val, bf_val, how="inner", predicate="overlaps")

print("5 casos de sobreposição LF × BF:\n")
amostra = joined[["NUMBLOCO_left", "AREA_left", "NUMBLOCO_right", "AREA_right"]].head(5)
amostra.columns = ["NUMBLOCO_LF", "AREA_LF", "NUMBLOCO_BF", "AREA_BF"]
print(amostra.to_string(index=False))

5 casos de sobreposição LF × BF:

 NUMBLOCO_LF      AREA_LF  NUMBLOCO_BF     AREA_BF
001636850000 38232.252691 007071270000 1298.538872
001636850000 38232.252691 007071260000 1310.556043
007025580000  8035.048352 001811430000  612.074176
007025580000  8035.048352 001811450000  788.450701
007025580000  8035.048352 001208870000  599.853973
